# Regularization

## Learning Objectives
1. Understand L1 and L2 penalty mechanics: how they affect weight magnitudes and sparsity.
2. Compare L1, L2, ElasticNet, and Dropout in PyTorch with OOM-safe training loops.
3. Apply AdamW (decoupled weight decay) vs Adam+L2 to understand regularization in modern optimizers.
4. Sweep regularization strength lambda to visualize the bias-variance trade-off on train/val curves.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge, Lasso, ElasticNet

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: L1 and L2 Penalty — NumPy Gradient Derivation

In [ ]:
# Show how L1 and L2 penalties modify the loss and gradient during SGD.
# L2: ||w||^2  -> gradient += 2*lambda*w  (weight decay, proportional to magnitude)
# L1: ||w||_1  -> gradient += lambda*sign(w)  (constant shrinkage, promotes sparsity)

def mse_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    return float(np.mean((y_pred - y_true) ** 2))


def regularized_gradient(
    X: np.ndarray, y: np.ndarray, w: np.ndarray,
    reg_type: str = 'l2', lam: float = 0.01
) -> np.ndarray:
    """Compute MSE gradient plus L1 or L2 regularization gradient."""
    y_pred = X @ w
    grad = 2 * X.T @ (y_pred - y) / len(y)
    if reg_type == 'l2':
        grad += 2 * lam * w
    elif reg_type == 'l1':
        grad += lam * np.sign(w)
    elif reg_type == 'elasticnet':
        grad += lam * np.sign(w) + lam * w  # Equal mix of L1 and L2
    return grad


# Synthetic regression: high-dimensional, many irrelevant features
rng = np.random.default_rng(0)
n_samples, n_features = 200, 50
# Only first 10 features are truly predictive
true_w = np.zeros(n_features)
true_w[:10] = rng.standard_normal(10)
X = rng.standard_normal((n_samples, n_features))
y = X @ true_w + rng.standard_normal(n_samples) * 0.5

lr = 0.01
n_steps = 500

print(f"{'Method':>12} | {'Final Loss':>12} | {'Zero Weights':>14} | {'L1 Norm':>9}")
print("-" * 60)

for reg_type in ['none', 'l2', 'l1', 'elasticnet']:
    w = rng.standard_normal(n_features) * 0.01
    for _ in range(n_steps):
        g = regularized_gradient(X, y, w, reg_type=reg_type if reg_type != 'none' else 'l2',
                                  lam=0.0 if reg_type == 'none' else 0.01)
        w -= lr * g
    loss = mse_loss(X @ w, y)
    zeros = np.sum(np.abs(w) < 1e-3)
    l1_norm = np.sum(np.abs(w))
    print(f"{reg_type:>12} | {loss:12.4f} | {zeros:14d} | {l1_norm:9.4f}")

## Level 2: L1/L2/ElasticNet/Dropout Comparison in PyTorch

In [ ]:
# Compare four regularization strategies on a binary classification MLP.
# Dropout: probabilistically zero neurons during training.
# L1/L2: add penalty to loss explicitly.

def l1_penalty(model: nn.Module) -> torch.Tensor:
    """Sum of absolute values of all parameters."""
    return sum(p.abs().sum() for p in model.parameters())


def l2_penalty(model: nn.Module) -> torch.Tensor:
    """Sum of squared values of all parameters."""
    return sum((p ** 2).sum() for p in model.parameters())


def build_mlp(dropout: float = 0.0) -> nn.Sequential:
    layers = [nn.Linear(50, 128), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(128, 64), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(64, 2))
    return nn.Sequential(*layers).to(device)


# Data
X_t = torch.tensor(X, dtype=torch.float32, device=device)
y_cls = torch.tensor((y > 0).astype(int), dtype=torch.long, device=device)
split = 160
train_ds = TensorDataset(X_t[:split], y_cls[:split])
val_ds = TensorDataset(X_t[split:], y_cls[split:])
tr_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
vl_loader = DataLoader(val_ds, batch_size=32)


def train_with_reg(
    reg_type: str, lam: float = 1e-3, dropout: float = 0.0, epochs: int = 20
) -> tuple:
    model = build_mlp(dropout)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            try:
                opt.zero_grad()
                loss = crit(model(xb), yb)
                if reg_type == 'l1':
                    loss += lam * l1_penalty(model)
                elif reg_type == 'l2':
                    loss += lam * l2_penalty(model)
                elif reg_type == 'elasticnet':
                    loss += lam * l1_penalty(model) + lam * l2_penalty(model)
                loss.backward()
                opt.step()
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    torch.cuda.empty_cache()
                    print(f"OOM during training with reg={reg_type}")
                    continue
                raise
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in vl_loader:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total, model


print(f"{'Method':>12} | {'Val Acc':>9}")
print("-" * 30)
for method, lam, drop in [
    ('none', 0.0, 0.0),
    ('l2', 1e-3, 0.0),
    ('l1', 1e-3, 0.0),
    ('elasticnet', 5e-4, 0.0),
    ('dropout', 0.0, 0.3),
]:
    acc, _ = train_with_reg(method, lam=lam, dropout=drop, epochs=20)
    print(f"{method:>12} | {acc:9.3f}")

## Real-World Example 1: AdamW vs Adam+L2 Regularization

In [ ]:
# AdamW decouples weight decay from the adaptive gradient scaling in Adam.
# In standard Adam+L2, L2 interacts with per-parameter learning rates,
# leading to under-regularization for parameters with high gradient variance.
# AdamW applies weight decay directly: w = w*(1-lr*decay) - lr*grad

def train_optimizer(
    optimizer_name: str, weight_decay: float = 1e-2, epochs: int = 30
) -> list:
    """Train MLP and return validation loss per epoch."""
    model = build_mlp(dropout=0.0)
    if optimizer_name == 'adamw':
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    elif optimizer_name == 'adam_l2':
        # Adam + L2: weight_decay is added to gradients (NOT decoupled)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    else:  # plain Adam
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0)

    crit = nn.CrossEntropyLoss()
    val_losses = []
    for _ in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = sum(crit(model(xb), yb).item() * len(yb) for xb, yb in vl_loader)
        val_losses.append(vl / len(vl_loader.dataset))
    return val_losses


results = {}
for opt_name in ['adam', 'adam_l2', 'adamw']:
    results[opt_name] = train_optimizer(opt_name, weight_decay=1e-2, epochs=30)

# Summary: compare final val loss
print(f"{'Optimizer':>12} | {'Final Val Loss':>16} | {'Min Val Loss':>12}")
print("-" * 48)
for name, hist in results.items():
    print(f"{name:>12} | {hist[-1]:16.4f} | {min(hist):12.4f}")

print("\nAdamW typically achieves lower final val loss by avoiding L2-adaptive interaction.")
print("Use AdamW for transformers; Adam+L2 for CNNs/RNNs where it's conventional.")

## Real-World Example 2: L1 Regularization for Feature Selection

In [ ]:
# L1 (Lasso) sets many weights exactly to zero — natural feature selection.
# Demonstrate how the number of non-zero coefficients varies with alpha.

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Rescale data for sklearn
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

print(f"True non-zero features: {(true_w != 0).sum()}  (features 0-9)")
print()
print(f"{'Alpha':>10} | {'Non-zero coefs':>16} | {'Val MSE':>10}")
print("-" * 44)

for alpha in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]:
    lasso = Lasso(alpha=alpha, max_iter=2000)
    lasso.fit(X_s[:160], y[:160])
    n_nonzero = (np.abs(lasso.coef_) > 1e-5).sum()
    preds = lasso.predict(X_s[160:])
    val_mse = np.mean((preds - y[160:]) ** 2)
    # Flag which of the true features were recovered
    recovered = np.where(np.abs(lasso.coef_[:10]) > 1e-5)[0]
    print(f"{alpha:10.3f} | {n_nonzero:16d} | {val_mse:10.4f} "
          f"  (recovered {len(recovered)}/10 true features)")

print("\nKey insight: alpha=0.01-0.05 recovers most true features")
print("while eliminating ~40 noise features. Alpha too high -> underfitting.")

## Real-World Example 3: Lambda Sweep — Train/Val Loss Curves

In [ ]:
# Sweep L2 regularization strength and record both training and validation loss.
# Visualize the classic U-shaped validation curve to find optimal lambda.

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

alphas = [1e-5, 1e-4, 1e-3, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
train_mses = []
val_mses = []

X_train_s = X_s[:160]
y_train = y[:160]
X_val_s = X_s[160:]
y_val = y[160:]

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_s, y_train)
    train_mses.append(mean_squared_error(y_train, ridge.predict(X_train_s)))
    val_mses.append(mean_squared_error(y_val, ridge.predict(X_val_s)))

best_idx = int(np.argmin(val_mses))
print(f"{'Alpha':>10} | {'Train MSE':>12} | {'Val MSE':>12}")
print("-" * 42)
for i, alpha in enumerate(alphas):
    marker = " <- best" if i == best_idx else ""
    print(f"{alpha:10.5f} | {train_mses[i]:12.4f} | {val_mses[i]:12.4f}{marker}")

print(f"\nOptimal alpha: {alphas[best_idx]} | Val MSE: {val_mses[best_idx]:.4f}")
print("\nTrade-off:")
print("  Low lambda  -> low train MSE, high val MSE (overfitting)")
print("  High lambda -> high train MSE, high val MSE (underfitting)")
print("  Sweet spot: lambda that minimizes the U-curve in val MSE.")